In [ ]:
import os, time
import pandas as pd
from dotenv import load_dotenv
from sodapy import Socrata

load_dotenv()
client = Socrata("data.cityofnewyork.us", os.getenv("SOCRATA_TOKEN"), timeout=120)

os.makedirs("ems_raw", exist_ok=True)

cols = "incident_datetime,zipcode,initial_severity_level_code,valid_incident_rspns_time_indc"
PAGE = 10000   # smaller pages are far less likely to time out into a 503

def get_page(where, offset, retries=8):
    for i in range(retries):
        try:
            return client.get("76xm-jjuj", select=cols, where=where,
                              order="incident_datetime", limit=PAGE, offset=offset)
        except Exception as e:
            wait = min(2 ** i, 60)
            print(f"  offset {offset}: {e} -> retry {i+1}/{retries} in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Gave up at offset {offset}")

for year in range(2016, 2023):
    final = f"ems_raw/{year}.csv"
    if os.path.exists(final):
        continue
    part_dir = f"ems_raw/{year}_parts"
    os.makedirs(part_dir, exist_ok=True)
    w = f"incident_datetime between '{year}-01-01T00:00:00' and '{year}-12-31T23:59:59'"

    offset = 0
    while True:
        part = f"{part_dir}/{offset:09d}.csv"
        if os.path.exists(part):                  # page already downloaded
            offset += PAGE
            continue
        batch = get_page(w, offset)
        if not batch:
            break
        pd.DataFrame.from_records(batch).to_csv(part, index=False)
        offset += PAGE
        print(f"{year}: {offset:,} rows requested")

    parts = sorted(os.listdir(part_dir))
    if parts:
        pd.concat((pd.read_csv(f"{part_dir}/{p}", dtype={"zipcode": str}) for p in parts),
                  ignore_index=True).to_csv(final, index=False)
        print(year, "saved")

2017: 10,000 rows requested
2017: 20,000 rows requested
2017: 30,000 rows requested
2017: 40,000 rows requested
2017: 50,000 rows requested
2017: 60,000 rows requested
2017: 70,000 rows requested
2017: 80,000 rows requested
2017: 90,000 rows requested
2017: 100,000 rows requested
2017: 110,000 rows requested
2017: 120,000 rows requested
2017: 130,000 rows requested
2017: 140,000 rows requested
2017: 150,000 rows requested
2017: 160,000 rows requested
2017: 170,000 rows requested
2017: 180,000 rows requested
2017: 190,000 rows requested
2017: 200,000 rows requested
2017: 210,000 rows requested
2017: 220,000 rows requested
2017: 230,000 rows requested
2017: 240,000 rows requested
2017: 250,000 rows requested
2017: 260,000 rows requested
2017: 270,000 rows requested
2017: 280,000 rows requested
2017: 290,000 rows requested
2017: 300,000 rows requested
2017: 310,000 rows requested
2017: 320,000 rows requested
2017: 330,000 rows requested
2017: 340,000 rows requested
2017: 350,000 rows requ

In [ ]:
import pandas as pd

EMS_YEARS = list(range(2016, 2023))          # a shorter list uses less memory and runs faster, e.g. [2021, 2022]

df = pd.concat((pd.read_csv(f"ems_raw/{y}.csv", dtype={"zipcode": str}) for y in EMS_YEARS), ignore_index=True)
df["incident_datetime"] = pd.to_datetime(df["incident_datetime"], errors="coerce")

# completeness check against the API's yearly totals: a short year silently biases everything downstream
API_TOTALS = {2016: 1441211, 2017: 1463446, 2018: 1530922, 2019: 1533251, 2020: 1412701, 2021: 1491454, 2022: 1583531}
got = df["incident_datetime"].dt.year.value_counts().sort_index()
check = pd.DataFrame({"rows": got, "api_total": pd.Series(API_TOTALS)}).loc[got.index]
check["pct_complete"] = (100 * check.rows / check.api_total).round(1)
print(df.shape); print(check)
short = check[check.pct_complete < 99].index.tolist()
if short:
    print(f"WARNING: {short} look incomplete. Delete ems_raw/<year>.csv and ems_raw/<year>_parts, then rerun the fetch cell.")

## Panel and features
One row per **ZIP code x 30-minute bucket**. A row is an *issue time* (the end of that bucket): `y`, `y_critical`, the lags and rolling sums (which include the current bucket) and the time features are all known by then.
Critical = initial severity code 1-3 (assumption F9, to be confirmed). Logic lives in `src/grease/forecast/features.py`, which is unit-tested.

In [ ]:
import sys
from pathlib import Path
import numpy as np

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
from grease.forecast import features as F

panel = F.build_panel(df)               # zipcode x 30-min bucket: y, y_critical, y_other
panel = F.add_history_features(panel)   # lags and rolling sums as of the issue bucket, plus time features
del df
print(panel.shape, panel.bucket.min(), "->", panel.bucket.max(), "| zipcodes:", panel.zipcode.nunique())

## Link all other data sources into one big table (`full`)
EMS counts are combined with everything fetched and cleaned by **`DarraghC.ipynb`** (run that notebook first; its outputs live in `notebooks/data/`):

| Joined on | From | Columns added |
|---|---|---|
| 30-minute `bucket` | `weather_calendar_30min.csv.gz` | temperature, feels-like, humidity, rain, wind, cloud, hot / cold / heat-stress flags, holidays, and the `eval_` heat / heavy-rain day flags |
| `zipcode` -> `zone` -> | `interim/zip_to_zone.csv` then `interim/zone_context.csv` | `zone`, `borough`, `lon`, `lat`, `pop_est`, `asthma_rate_per_10k`, Census columns, hospitals / firehouses in the zone, distance to nearest hospital |

`full` is the modelling table. **Which columns the model uses is an explicit list** (`FEATURE_GROUPS` below, decision M2), never "everything in the table".
`eval_*` columns only slice results by heat or heavy-rain days and are never model inputs.

In [ ]:
from pathlib import Path

DATA = Path("data")                                   # notebooks/data, written by DarraghC.ipynb
needed = {"weather": DATA / "weather_calendar_30min.csv.gz",
          "zip_to_zone": DATA / "interim" / "zip_to_zone.csv",
          "zone_context": DATA / "interim" / "zone_context.csv"}
missing = [str(f) for f in needed.values() if not f.exists()]
assert not missing, f"Run DarraghC.ipynb first, it creates: {missing}"

wx = pd.read_csv(needed["weather"], parse_dates=["bucket"]).set_index("bucket")
zip_to_zone = pd.read_csv(needed["zip_to_zone"], dtype=str).set_index("zipcode")["zone"]
zone_ctx = pd.read_csv(needed["zone_context"], dtype={"zone": str}).set_index("zone").drop(columns=["label"])

full = panel                                          # no copy: the table is tens of millions of rows
del panel
base_cols = list(full.columns)

# 1) EMS zipcode -> zone (look up once per ZIP, not once per row)
zips = full["zipcode"].cat.categories
zone_of_zip = pd.Series(zips, index=zips).map(zip_to_zone)
zone_names = pd.Index(sorted(zone_of_zip.dropna().unique()))
zip_to_code = np.array([zone_names.get_loc(z) if pd.notna(z) else -1 for z in zone_of_zip], dtype="int16")
zone_code = zip_to_code[full["zipcode"].cat.codes.to_numpy()]            # -1 = ZIP is not in any zone
full["zone"] = pd.Categorical.from_codes(zone_code, categories=zone_names)

# 2) zone context (borough, health context, access to care, ...)
for c in zone_ctx.columns:
    vals = zone_ctx[c].reindex(zone_names)
    if pd.api.types.is_numeric_dtype(vals):
        arr = vals.to_numpy(dtype="float32")
        full[c] = np.where(zone_code >= 0, arr[zone_code], np.float32("nan")).astype("float32")
    else:                                             # text column such as borough -> category
        cats = pd.Index(sorted(vals.dropna().unique()))
        lut = np.array([cats.get_loc(v) if pd.notna(v) else -1 for v in vals], dtype="int16")
        full[c] = pd.Categorical.from_codes(np.where(zone_code >= 0, lut[zone_code], -1), categories=cats)

# 3) weather + calendar on the 30-minute bucket
pos = wx.index.get_indexer(full["bucket"])            # -1 where a bucket has no weather row
for c in wx.columns:
    arr = wx[c].to_numpy(dtype="float32")
    full[c] = np.where(pos >= 0, arr[pos], np.float32("nan")).astype("float32")

EVAL_COLS = [c for c in full.columns if c.startswith("eval_")]
print("full:", full.shape, f"| ~{full.memory_usage().sum() / 1e9:.1f} GB in memory")
print(f"rows whose ZIP is in no zone: {(zone_code < 0).mean():.2%} | rows with no weather: {(pos < 0).mean():.2%}")
print("slice results by these, never use as model inputs:", EVAL_COLS)
print("added columns:", [c for c in full.columns if c not in base_cols])
full.head(3).T

## Model: expected demand for the next 30 min / 1 h / 2 h / 3 h (decisions D8-D10)
For every ZIP and every issue time we predict the **expected number of incidents in the next window**, separately for **critical** and **other** calls (8 LightGBM Poisson models: 2 acuities x 4 horizons).
The risk index (further down) is a derived, explainable layer on top, not part of the model. Code: `src/grease/forecast/` (`lgbm.py`, `baseline.py`, `evaluate.py`, `zones.py`) and `src/grease/risk.py`.

- **Split by date** (F11): train up to the validation start, validate for early stopping, test on 2022. A gap of 3 hours is dropped before each boundary because a training target looks up to 3 hours ahead.
- **Baselines** (M4), scored on exactly the same rows: historical mean for that ZIP x weekday x time of day; that mean x a citywide recent-surge multiplier; and the same window one week earlier.
- **Settings** below are the knobs for the experiments in `docs/decisions.md` (section 4).

In [ ]:
from grease.forecast import baseline as B, evaluate as E, zones as Z
from grease.forecast.lgbm import ForecastModel
from grease import risk

TEST_START = pd.Timestamp("2022-01-01")     # F11: test on 2022
VAL_START = pd.Timestamp("2021-10-01")      # last quarter of 2021 = validation set (early stopping)
TRAIN_START = pd.Timestamp("2016-01-01")
EXCLUDE_YEARS = []                          # F10: e.g. [2020] to drop COVID from training
FEATURE_GROUPS = F.DEFAULT_GROUPS           # F5-F7: also try adding "weather_extra", "zone_census", "zone_health_access"
PARAMS = {}                                 # LightGBM overrides, e.g. {"n_estimators": 300, "learning_rate": 0.1} for a quick run
print("feature groups:", FEATURE_GROUPS)

In [ ]:
full = F.add_targets(full)       # M1 / M5: incidents in the next 30m / 1h / 2h / 3h, for critical, other and all
features = F.select_features(full.columns, FEATURE_GROUPS)
print(len(features), "features:", features)

embargo = pd.Timedelta(minutes=30 * max(F.HORIZONS.values()))     # keep a training target from reaching into the next period
in_train = (full.bucket >= TRAIN_START) & (full.bucket < VAL_START - embargo) & ~full.bucket.dt.year.isin(EXCLUDE_YEARS)
in_valid = (full.bucket >= VAL_START) & (full.bucket < TEST_START - embargo)
in_test = full.bucket >= TEST_START
for name, m in (("train", in_train), ("validation", in_valid), ("test", in_test)):
    assert m.any(), f"{name} period is empty: the loaded EMS_YEARS do not cover it. Adjust TRAIN_START / VAL_START / TEST_START above."
    print(f"{name:10s} {m.sum():>12,} rows   {full.bucket[m].min()} -> {full.bucket[m].max()}")
train, valid, test = full[in_train], full[in_valid], full[in_test]

In [ ]:
model = ForecastModel(features, PARAMS).fit(train, valid)

In [ ]:
pred = model.predict(test)                 # pred_critical_30m ... pred_other_3h, pred_all_30m ... (expected counts, not integers)
surge = B.surge_multiplier(train, test)
results = {}                                # horizon -> actuals and every model's predictions, on the same rows
for h in F.HORIZONS:
    t = F.target_name("all", h)
    hist = B.historical_mean(train, test, t)
    results[h] = dict(actual=test[t].to_numpy(), preds={
        "model": pred[f"pred_all_{h}"].to_numpy(), "hist_mean": hist, "hist_x_surge": hist * surge,
        "last_week": B.last_week(full, t).reindex(test.index).to_numpy()})
print("predicted", f"{len(pred):,}", "test rows for horizons", list(F.HORIZONS))

In [ ]:
slices = {"all test rows": np.ones(len(test), bool),
          "heat days (daily max >= 32C)": (test.eval_is_heat_day == 1).to_numpy(),
          "heavy-rain days": (test.eval_is_heavy_rain_day == 1).to_numpy(),
          "busiest 20 zipcodes": test.zipcode.isin(E.busiest_zipcodes(train, 20)).to_numpy()}
for b in sorted(test.borough.dropna().unique()):
    slices[f"borough: {b}"] = (test.borough == b).to_numpy()

scores = pd.concat([E.score_slices(r["actual"], r["preds"], slices).assign(horizon=h) for h, r in results.items()], ignore_index=True)
scores["horizon"] = pd.Categorical(scores["horizon"], categories=list(F.HORIZONS), ordered=True)

mae = scores.pivot_table(index=["slice", "horizon"], columns="model", values="MAE", observed=True)
mae["model beats best baseline by (%)"] = (100 * (1 - mae["model"] / mae[["hist_mean", "hist_x_surge", "last_week"]].min(axis=1))).round(1)
print("MAE of the expected count (lower is better). Positive % = the model has lower error than the best baseline.")
display(mae.round(4))
print("Poisson deviance (lower is better):")
display(scores.pivot_table(index=["slice", "horizon"], columns="model", values="deviance", observed=True).round(4))
print("Bias of the model (mean predicted / mean actual; 1.0 = unbiased), per slice and horizon:")
display(scores[scores.model == "model"].pivot_table(index="slice", columns="horizon", values="bias", observed=True).round(3))

In [ ]:
for h in ("1h", "3h"):
    print(f"Calibration, next {h}, all calls: mean predicted vs mean actual by prediction decile (ratio should be close to 1 in every row)")
    display(E.calibration_table(results[h]["actual"], results[h]["preds"]["model"]).round(4))
imp = model.feature_importance()
print("Feature importance (share of total gain, averaged over the 8 models):")
display(imp.mean(axis=1).sort_values(ascending=False).head(15).round(3))

In [ ]:
zip_to_zone = pd.read_csv(DATA / "interim" / "zip_to_zone.csv", dtype=str).set_index("zipcode")["zone"]
zone_ctx = pd.read_csv(DATA / "interim" / "zone_context.csv", dtype={"zone": str}).set_index("zone")

t0 = test.groupby("bucket")["y"].sum().idxmax()                       # the busiest half-hour in the test period
rows = test.index[test.bucket == t0]
snap = pred.loc[rows].assign(zipcode=test.loc[rows, "zipcode"].to_numpy(), bucket=t0)
zones_now = Z.aggregate_to_zones(snap, zip_to_zone, ["pred_critical_1h", "pred_other_1h"]).set_index("zone")   # M7: ZIP -> zone, exact sum
zones_now["risk_index"] = risk.risk_index(zones_now.pred_critical_1h, zones_now.pred_other_1h, risk.vulnerability(zone_ctx)).round(1)
zones_now = zones_now.join(zone_ctx[["borough"]]).sort_values("risk_index", ascending=False)
print("Snapshot at", t0, "- expected incidents in the next hour per zone, and the 0-100 risk index (D9)")
zones_now.head(10).round(2)

In [ ]:
out = DATA / "processed"
out.mkdir(parents=True, exist_ok=True)
model.save(out / "forecast_models")             # LightGBM text files + meta.json: portable to the VM
scores.to_csv(out / "forecast_scores.csv", index=False)
print("saved models and scores to", out)